# Solutions — Statistical Aggregations (Medium 20)

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

lineitem     = spark.table("samples.tpch.lineitem")
transactions = spark.table("samples.bakehouse.sales_transactions")
trips        = spark.table("samples.nyctaxi.trips")

## Solution 1 — Exact vs Approx Distinct Count

In [ ]:
result_1 = transactions.agg(
    F.countDistinct("customerID").alias("exact_distinct"),
    F.approx_count_distinct("customerID").alias("approx_distinct")
)

## Solution 2 — Stddev, Variance, Population Stats per Franchise

In [ ]:
result_2 = (
    transactions
    .groupBy("franchiseID")
    .agg(
        F.round(F.mean("totalPrice"), 2).alias("mean_price"),
        F.round(F.stddev_samp("totalPrice"), 4).alias("std_price"),
        F.round(F.var_samp("totalPrice"), 4).alias("var_price"),
        F.round(F.stddev_pop("totalPrice"), 4).alias("std_pop"),
        F.round(F.var_pop("totalPrice"), 4).alias("var_pop")
    )
    .orderBy("franchiseID")
)

## Solution 3 — Correlation and Covariance

In [ ]:
result_3 = transactions.agg(
    F.round(F.corr("quantity","totalPrice"), 4).alias("correlation_qty_price"),
    F.round(F.covar_pop("quantity","totalPrice"), 4).alias("covar_pop_qty_price"),
    F.round(F.covar_samp("quantity","totalPrice"), 4).alias("covar_samp_qty_price")
)

## Solution 4 — Percentile Approximation

In [ ]:
pcts = trips.agg(
    F.percentile_approx("fare_amount", [0.25, 0.5, 0.75]).alias("pcts"),
    F.percentile_approx("fare_amount", [i/10.0 for i in range(1,11)]).alias("decile_array")
).collect()[0]
result_4 = spark.createDataFrame([(
    float(pcts["pcts"][0]), float(pcts["pcts"][1]), float(pcts["pcts"][2]),
    [float(v) for v in pcts["decile_array"]]
)], ["p25","p50","p75","decile_array"])

## Solution 5 — Skewness and Kurtosis

In [ ]:
result_5 = spark.createDataFrame([
    ("totalPrice",
     float(transactions.agg(F.skewness("totalPrice")).collect()[0][0]),
     float(transactions.agg(F.kurtosis("totalPrice")).collect()[0][0])),
    ("fare_amount",
     float(trips.agg(F.skewness("fare_amount")).collect()[0][0]),
     float(trips.agg(F.kurtosis("fare_amount")).collect()[0][0])),
], ["column_name","skewness_val","kurtosis_val"])

## Solution 6 — Rollup (Subtotals at 3 Levels)

In [ ]:
result_6 = (
    transactions
    .rollup("franchiseID","product")
    .agg(
        F.round(F.sum("totalPrice"), 2).alias("revenue"),
        F.count("*").alias("txn_count")
    )
    .orderBy("franchiseID","product")
)

## Solution 7 — Cube (All Combination Subtotals)

In [ ]:
result_7 = (
    transactions
    .cube("franchiseID","product","paymentMethod")
    .agg(F.round(F.sum("totalPrice"), 2).alias("revenue"))
    .withColumn("grouping_level",
        F.concat_ws("-",
            F.when(F.grouping("franchiseID")==1, F.lit("ALL")).otherwise(F.col("franchiseID").cast("string")),
            F.when(F.grouping("product")==1, F.lit("ALL")).otherwise(F.col("product")),
            F.when(F.grouping("paymentMethod")==1, F.lit("ALL")).otherwise(F.col("paymentMethod"))
        )
    )
    .orderBy("franchiseID","product","paymentMethod")
)

## Solution 8 — Collect, Sort, Slice for Top-N

In [ ]:
result_8 = (
    transactions
    .groupBy("franchiseID")
    .agg(F.sort_array(F.collect_list("totalPrice"), asc=False).alias("all_prices_sorted"))
    .withColumn("top_5_prices", F.slice("all_prices_sorted", 1, 5))
    .withColumn("min_price",    F.array_min("all_prices_sorted"))
    .withColumn("max_price",    F.array_max("all_prices_sorted"))
    .select("franchiseID","all_prices_sorted","top_5_prices","min_price","max_price")
)